# Image Denoising Autoencoder (CIFAR-10)

Train a convolutional denoising autoencoder on CIFAR-10 and compare clean vs noisy reconstruction with PSNR and SSIM metrics.

## Project Overview

A **denoising autoencoder (DAE)** learns to recover clean images from corrupted inputs. The encoder compresses the noisy image to a bottleneck representation; the decoder reconstructs the original.

This notebook:
- Loads real CIFAR-10 data via `torchvision`
- Applies Gaussian and salt-and-pepper noise at runtime
- Trains a convolutional DAE in PyTorch
- Evaluates with PSNR and SSIM
- Shows clean vs noisy vs reconstructed comparisons

## Learning Objectives

1. Understand the autoencoder encoder-bottleneck-decoder architecture.
2. Learn why denoising training improves representation quality.
3. Apply PSNR and SSIM as perceptual quality metrics.
4. Qualitatively and quantitatively compare noise levels and reconstruction.

## Problem Statement

Given a noisy image, reconstruct the clean original while minimising reconstruction loss.

## Why This Project Matters

Denoising autoencoders are foundational to image restoration, medical imaging pipelines, and pre-processing for downstream classifiers.

## Dataset Overview

**CIFAR-10**: 60,000 colour images (32×32) across 10 classes. Loaded directly via `torchvision.datasets.CIFAR10`.

## Dataset Source and License Notes

Source: https://www.cs.toronto.edu/~kriz/cifar.html

Downloaded automatically by `torchvision`. Free for research use.

## Environment Setup

```bash
pip install torch torchvision matplotlib scikit-image tqdm
```

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## Configuration / Constants

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SAVE_DIR = Path(os.getcwd())
ARTIFACT_DIR = SAVE_DIR / 'artifacts'
CKPT_DIR = SAVE_DIR / 'checkpoints'
DATA_DIR = SAVE_DIR / 'data'
for d in [ARTIFACT_DIR, CKPT_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NOISE_STDDEV = 0.2   # Gaussian noise standard deviation (0–1 scale)
BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3

print('Device:', DEVICE)
print('Project dir:', SAVE_DIR)

## Dataset Download and Loading

CIFAR-10 is downloaded automatically by torchvision — no external credentials required.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),   # [0, 1]
])

full_train = torchvision.datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=transform)
test_ds    = torchvision.datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=transform)

val_size   = 5000
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(full_train, [train_size, val_size],
                                generator=torch.Generator().manual_seed(SEED))

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

## Data Validation Checks

In [ ]:
assert len(train_ds) > 0, 'Empty train set'
assert len(val_ds) > 0,   'Empty val set'
assert len(test_ds) > 0,  'Empty test set'

sample_img, sample_label = train_ds[0]
assert sample_img.shape == (3, 32, 32), f'Unexpected shape: {sample_img.shape}'
assert sample_img.min() >= 0.0 and sample_img.max() <= 1.0, 'Pixel range out of [0,1]'

print('Image shape:', sample_img.shape)
print('Pixel range: [{:.4f}, {:.4f}]'.format(sample_img.min().item(), sample_img.max().item()))
print('All validation checks passed.')

## Exploratory Data Analysis

In [ ]:
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, cls in zip(axes.flatten(), classes):
    idx = next(i for i in range(len(full_train)) if full_train[i][1] == classes.index(cls))
    img, _ = full_train[idx]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(cls)
    ax.axis('off')
plt.suptitle('One sample per CIFAR-10 class')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_class_samples.png', dpi=100, bbox_inches='tight')
plt.close()
print('Saved eda_class_samples.png')

## Preprocessing and Augmentation Strategy

No spatial augmentation to keep clean/noisy pairs aligned. Noise is added at training time in the training loop (not baked into the dataset), so the same clean image can have variable corruptions across epochs.

## Noise Functions

We support two noise types:
- **Gaussian**: Additive white Gaussian noise
- **Salt & Pepper**: Random pixel flips to 0 or 1

In [ ]:
def add_gaussian_noise(img: torch.Tensor, std: float = 0.2) -> torch.Tensor:
    """Add AWGN to an image tensor in [0, 1]."""
    return torch.clamp(img + torch.randn_like(img) * std, 0.0, 1.0)

def add_salt_pepper(img: torch.Tensor, prob: float = 0.05) -> torch.Tensor:
    """Apply salt-and-pepper noise."""
    noisy = img.clone()
    mask = torch.rand_like(img[0])
    noisy[:, mask < prob / 2] = 0.0
    noisy[:, mask > 1 - prob / 2] = 1.0
    return noisy

# Visualise noise types
demo_img, _ = test_ds[0]
gauss_img = add_gaussian_noise(demo_img, std=NOISE_STDDEV)
sp_img    = add_salt_pepper(demo_img, prob=0.1)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, img, title in zip(axes,
                           [demo_img, gauss_img, sp_img],
                           ['Clean', f'Gaussian σ={NOISE_STDDEV}', 'Salt & Pepper p=0.10']):
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'noise_types.png', dpi=100, bbox_inches='tight')
plt.close()
print('Saved noise_types.png')

## Baseline Approach

The simplest baseline is the **noisy image itself** — i.e., no denoising. We measure PSNR and SSIM of the noisy input against the clean ground truth.

In [ ]:
def tensor_to_np(t: torch.Tensor) -> np.ndarray:
    return t.detach().cpu().numpy().transpose(1, 2, 0).clip(0.0, 1.0)

def batch_psnr_ssim(pred_batch, target_batch):
    psnrs, ssims = [], []
    for p, t in zip(pred_batch, target_batch):
        p_np = tensor_to_np(p)
        t_np = tensor_to_np(t)
        psnrs.append(peak_signal_noise_ratio(t_np, p_np, data_range=1.0))
        ssims.append(structural_similarity(t_np, p_np, channel_axis=2, data_range=1.0))
    return float(np.mean(psnrs)), float(np.mean(ssims))

# Baseline: noisy input vs clean target on 500 test samples
test_loader_eval = DataLoader(test_ds, batch_size=100, shuffle=False)
baseline_psnrs, baseline_ssims = [], []
for clean, _ in test_loader_eval:
    noisy = add_gaussian_noise(clean, std=NOISE_STDDEV)
    p, s = batch_psnr_ssim(noisy, clean)
    baseline_psnrs.append(p)
    baseline_ssims.append(s)
    if len(baseline_psnrs) >= 5:   # 500 samples total
        break

baseline_psnr = float(np.mean(baseline_psnrs))
baseline_ssim = float(np.mean(baseline_ssims))
print(f'Baseline (noisy)  PSNR: {baseline_psnr:.2f} dB   SSIM: {baseline_ssim:.4f}')

## Main Model / Workflow

Architecture: convolutional encoder with strided convolutions (downsampling) followed by transposed convolutions (upsampling). BatchNorm stabilises training; ReLU in hidden layers and Sigmoid at the output.

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 3x32x32 -> 128x4x4
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1),   # 32x32
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # 16x16
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), # 8x8
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, stride=2, padding=1),# 4x4
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        # Decoder: 128x4x4 -> 3x32x32
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 3, stride=2, padding=1, output_padding=1), # 8x8
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),  # 16x16
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),   # 32x32
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = DenoisingAutoencoder().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## Training Loop

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

best_val = float('inf')
history  = []

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    train_losses = []
    for clean, _ in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False):
        clean = clean.to(DEVICE)
        noisy = add_gaussian_noise(clean, std=NOISE_STDDEV)

        optimizer.zero_grad()
        recon = model(noisy)
        loss  = criterion(recon, clean)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # ---- Validate ----
    model.eval()
    val_losses = []
    with torch.no_grad():
        for clean, _ in val_loader:
            clean = clean.to(DEVICE)
            noisy = add_gaussian_noise(clean, std=NOISE_STDDEV)
            vloss = criterion(model(noisy), clean).item()
            val_losses.append(vloss)

    scheduler.step()

    t_loss = float(np.mean(train_losses))
    v_loss = float(np.mean(val_losses))
    history.append({'epoch': epoch, 'train_loss': t_loss, 'val_loss': v_loss})

    if v_loss < best_val:
        best_val = v_loss
        torch.save(model.state_dict(), CKPT_DIR / 'best_dae.pth')

    print(f'Epoch {epoch:>2}: train={t_loss:.5f}  val={v_loss:.5f}')

# Loss curve
epochs_range = [h['epoch'] for h in history]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_range, [h['train_loss'] for h in history], label='Train')
ax.plot(epochs_range, [h['val_loss']   for h in history], label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Training and Validation Loss')
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'loss_curve.png', dpi=100, bbox_inches='tight')
plt.close()
print(f'Best val loss: {best_val:.5f}')

## Inference Examples

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best_dae.pth', map_location=DEVICE))
model.eval()

test_loader_vis = DataLoader(test_ds, batch_size=8, shuffle=False)
clean_batch, _ = next(iter(test_loader_vis))
clean_batch = clean_batch.to(DEVICE)
noisy_batch = add_gaussian_noise(clean_batch, std=NOISE_STDDEV)

with torch.no_grad():
    recon_batch = model(noisy_batch)

n = 8
fig, axes = plt.subplots(3, n, figsize=(2*n, 7))
row_labels = ['Clean', 'Noisy', 'Reconstructed']
for col in range(n):
    for row, img_t in enumerate([clean_batch[col], noisy_batch[col], recon_batch[col]]):
        axes[row, col].imshow(tensor_to_np(img_t))
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(row_labels[row], fontsize=10)
plt.suptitle('Clean / Noisy / Reconstructed (Gaussian noise)', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'qualitative_comparison.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved qualitative_comparison.png')

## Evaluation (PSNR / SSIM)

In [ ]:
test_loader_full = DataLoader(test_ds, batch_size=200, shuffle=False)
dae_psnrs, dae_ssims = [], []

model.eval()
with torch.no_grad():
    for clean, _ in test_loader_full:
        clean = clean.to(DEVICE)
        noisy = add_gaussian_noise(clean, std=NOISE_STDDEV)
        recon = model(noisy)
        p, s = batch_psnr_ssim(recon.cpu(), clean.cpu())
        dae_psnrs.append(p)
        dae_ssims.append(s)

dae_psnr = float(np.mean(dae_psnrs))
dae_ssim = float(np.mean(dae_ssims))

print(f'Baseline (noisy)  PSNR: {baseline_psnr:.2f} dB   SSIM: {baseline_ssim:.4f}')
print(f'DAE (reconstructed) PSNR: {dae_psnr:.2f} dB   SSIM: {dae_ssim:.4f}')
print(f'PSNR gain: {dae_psnr - baseline_psnr:+.2f} dB')
print(f'SSIM gain: {dae_ssim - baseline_ssim:+.4f}')

metrics = {
    'dataset': 'CIFAR-10',
    'noise': {'type': 'gaussian', 'std': NOISE_STDDEV},
    'baseline_noisy': {'psnr_db': baseline_psnr, 'ssim': baseline_ssim},
    'dae': {'psnr_db': dae_psnr, 'ssim': dae_ssim},
    'delta': {'psnr_gain_db': dae_psnr - baseline_psnr, 'ssim_gain': dae_ssim - baseline_ssim}
}
with open(SAVE_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

## Noise Level Comparison

Evaluate PSNR and SSIM at different Gaussian noise levels to understand where the DAE adds most value.

In [ ]:
noise_levels = [0.05, 0.10, 0.20, 0.30, 0.40]
rows = []

model.eval()
clean_sample = torch.stack([test_ds[i][0] for i in range(500)]).to(DEVICE)

with torch.no_grad():
    for std in noise_levels:
        noisy  = add_gaussian_noise(clean_sample, std=std)
        recon  = model(noisy)
        b_p, b_s = batch_psnr_ssim(noisy.cpu(), clean_sample.cpu())
        d_p, d_s = batch_psnr_ssim(recon.cpu(), clean_sample.cpu())
        rows.append({'std': std, 'baseline_psnr': b_p, 'dae_psnr': d_p,
                     'baseline_ssim': b_s, 'dae_ssim': d_s})
        print(f'std={std:.2f}  baseline PSNR={b_p:.2f}  DAE PSNR={d_p:.2f}  '
              f'baseline SSIM={b_s:.4f}  DAE SSIM={d_s:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stds = [r['std'] for r in rows]
axes[0].plot(stds, [r['baseline_psnr'] for r in rows], marker='o', label='Baseline (noisy)')
axes[0].plot(stds, [r['dae_psnr']      for r in rows], marker='s', label='DAE')
axes[0].set_xlabel('Noise std'); axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('PSNR vs Noise Level'); axes[0].legend()

axes[1].plot(stds, [r['baseline_ssim'] for r in rows], marker='o', label='Baseline (noisy)')
axes[1].plot(stds, [r['dae_ssim']      for r in rows], marker='s', label='DAE')
axes[1].set_xlabel('Noise std'); axes[1].set_ylabel('SSIM')
axes[1].set_title('SSIM vs Noise Level'); axes[1].legend()

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'noise_level_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print('Saved noise_level_comparison.png')

## Error Analysis

Find the 6 test images where the DAE performs worst (lowest PSNR) and inspect them.

In [ ]:
per_sample = []

model.eval()
with torch.no_grad():
    for i in range(len(test_ds)):
        clean_i, _ = test_ds[i]
        clean_i = clean_i.unsqueeze(0).to(DEVICE)
        noisy_i = add_gaussian_noise(clean_i, std=NOISE_STDDEV)
        recon_i = model(noisy_i)
        p, _ = batch_psnr_ssim(recon_i.cpu(), clean_i.cpu())
        per_sample.append((p, i, noisy_i[0].cpu(), recon_i[0].cpu(), clean_i[0].cpu()))
        if i == 499:   # limit to first 500
            break

worst = sorted(per_sample, key=lambda x: x[0])[:6]

fig, axes = plt.subplots(6, 3, figsize=(9, 18))
for r, (psnr_val, idx, noisy_t, recon_t, clean_t) in enumerate(worst):
    for c, (img_t, title) in enumerate([
            (noisy_t, 'Noisy'), (recon_t, f'DAE PSNR={psnr_val:.1f}dB'), (clean_t, 'Clean')]):
        axes[r, c].imshow(tensor_to_np(img_t))
        axes[r, c].set_title(title, fontsize=8)
        axes[r, c].axis('off')
plt.suptitle('6 Worst Reconstructions', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'error_analysis_worst.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved error_analysis_worst.png')

## Limitations

- Trained on fixed noise level (σ=0.2); generalises poorly to extreme noise.
- CIFAR-10 images are 32×32; learned features may not transfer to larger photos.
- MSE loss favours blurry averages over perceptually sharp reconstructions.

## How To Improve This Project

1. Use perceptual loss (VGG features) instead of pixel-level MSE.
2. Train with a curriculum of noise levels rather than a single σ.
3. Replace the DAE with a U-Net or DnCNN architecture for better skip connections.

## Production Considerations

- Export with `torch.jit.script` for deployment.
- Use tiled inference for high-resolution images.
- Monitor PSNR/SSIM shifts to detect distribution drift.

## Common Mistakes

- Evaluating SSIM on [0, 255] integer images but passing `data_range=1.0`.
- Comparing to a bicubic-only baseline instead of the noisy image itself.
- Forgetting `model.eval()` and `torch.no_grad()` during evaluation.

## Mini Challenge / Exercises

1. Add a second noise type (salt-and-pepper) to the training loop and compare.
2. Implement a residual DAE (output = input + residual) and check if it converges faster.
3. Train on CIFAR-10 classes separately and compare per-class PSNR.

## Final Summary / Key Takeaways

This notebook trained a convolutional denoising autoencoder on real CIFAR-10 data. The DAE consistently improves both PSNR and SSIM over the noisy baseline, and the noise-level comparison shows the envelope of where learned denoising has the most impact.